# ProteinMPNN 서열 설계 노트북 (Windows / CPU)

RFdiffusion으로 생성한 binder backbone(PDB)에 ProteinMPNN으로 서열을 설계합니다.

**사전 준비 (VSCode 터미널에서 1회):**
```bash
git clone https://github.com/dauparas/ProteinMPNN.git
conda create -n mpnn python=3.10 -y
conda activate mpnn
conda install pytorch cpuonly -c pytorch -y
pip install numpy ipykernel
python -m ipykernel install --user --name mpnn --display-name "Python (mpnn)"
```
설치 후 이 노트북의 커널을 **"Python (mpnn)"** 으로 선택하세요.


## 1. 경로 및 파라미터 설정
경로 3개만 본인 환경에 맞게 수정하세요.

In [ ]:
import os, subprocess, sys

# ── 본인 환경에 맞게 수정 ──
MPNN_DIR = r"C:\Users\82105\Desktop\ProteinMPNN"                     # ProteinMPNN 레포 (protein_mpnn_run.py 있는 곳)
PDB_DIR  = r"C:\Users\82105\Desktop\design\ProteinMPNN\Input_pdb"    # RFdiffusion 출력 PDB 폴더
OUT_DIR  = r"C:\Users\82105\Desktop\design\ProteinMPNN\Output_pdb"   # 결과 저장 폴더

# ── 설계 파라미터 ──
DESIGN_CHAINS = "B"     # chain A: CTHRC1(214 res, 고정) / chain B: binder(10 res, 설계)
NUM_SEQ       = 4       # backbone 1개당 생성할 서열 수
SAMPLING_TEMP = "0.1"   # binder design 권장 (낮을수록 보수적)
SEED          = "37"

os.makedirs(OUT_DIR, exist_ok=True)
print("MPNN_DIR :", MPNN_DIR)
print("PDB_DIR  :", PDB_DIR)
print("OUT_DIR  :", OUT_DIR)

## 2. PDB chain 확인
binder가 정말 chain A인지 눈으로 확인합니다. 만약 binder가 다른 chain이면 위 `DESIGN_CHAINS`를 고치세요.

In [ ]:
import glob

pdbs = sorted(glob.glob(os.path.join(PDB_DIR, "*.pdb")))
assert pdbs, f"PDB 파일이 없습니다: {PDB_DIR}"
print(f"총 {len(pdbs)}개 PDB 발견. 첫 파일 chain/잔기수 확인:\n")

# 첫 PDB의 chain별 잔기 수 세기 (CA 기준)
chains = {}
with open(pdbs[0]) as f:
    for line in f:
        if line.startswith("ATOM") and line[12:16].strip() == "CA":
            ch = line[21]
            chains[ch] = chains.get(ch, 0) + 1

print("파일:", os.path.basename(pdbs[0]))
for ch, n in chains.items():
    print(f"  chain {ch}: {n} residues")
print(f"\n→ 설계 대상으로 지정한 chain: {DESIGN_CHAINS}")
print("  (binder는 보통 짧은 쪽. CTHRC1 target보다 잔기 수가 작아야 맞습니다)")


## 3. PDB 파싱
폴더 안 모든 PDB를 ProteinMPNN 입력 형식(jsonl)으로 변환합니다.

In [ ]:
helper = os.path.join(MPNN_DIR, "helper_scripts")
parsed = os.path.join(OUT_DIR, "parsed.jsonl")

subprocess.run([
    sys.executable, os.path.join(helper, "parse_multiple_chains.py"),
    "--input_path", PDB_DIR,
    "--output_path", parsed,
], check=True)
print("파싱 완료 →", parsed)


## 4. 설계할 chain 지정
binder chain만 설계하고, 나머지(CTHRC1 target) 서열은 고정합니다.

In [ ]:
assigned = os.path.join(OUT_DIR, "assigned.jsonl")

subprocess.run([
    sys.executable, os.path.join(helper, "assign_fixed_chains.py"),
    "--input_path", parsed,
    "--output_path", assigned,
    "--chain_list", DESIGN_CHAINS,
], check=True)
print("chain 지정 완료 →", assigned)


## 5. ProteinMPNN 실행
CPU에서 실행됩니다. backbone 수가 많으면 시간이 걸립니다 (파일럿은 수십~수백 개 권장).

In [ ]:
subprocess.run([
    sys.executable, os.path.join(MPNN_DIR, "protein_mpnn_run.py"),
    "--jsonl_path",         parsed,
    "--chain_id_jsonl",     assigned,
    "--out_folder",         OUT_DIR,
    "--num_seq_per_target", str(NUM_SEQ),
    "--sampling_temp",      SAMPLING_TEMP,
    "--seed",               SEED,
    "--batch_size",         "1",   # CPU는 1 권장
], check=True)

print("\n완료! 결과 FASTA 위치:", os.path.join(OUT_DIR, "seqs"))


## 6. 결과 미리보기
생성된 서열과 점수(score, global_score, seq_recovery)를 확인합니다.

In [ ]:
seq_dir = os.path.join(OUT_DIR, "seqs")
fastas = sorted(glob.glob(os.path.join(seq_dir, "*.fa")) + glob.glob(os.path.join(seq_dir, "*.fasta")))
print(f"FASTA 파일 {len(fastas)}개 생성됨\n")

# 첫 파일 내용 일부 출력
if fastas:
    print("=== 예시:", os.path.basename(fastas[0]), "===")
    with open(fastas[0]) as f:
        print(f.read()[:1500])
